In [ ]:
"""Data Visualization Playground."""
# Load Packages
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm

from statsmodels.iolib.summary2 import summary_col

In [ ]:
# Define a function to clean and recode political knowledge responses
def clean_knowledge_variable(series, correct_values):
    # Replace invalid codes with NaN
    series_cleaned = series.replace([-9, -5, -4, -1], pd.NA)
    # Recode correct answers as 1, others as 0
    return series_cleaned.isin(correct_values).astype(int) 

In [ ]:
# Data Prep
data_url = "https://raw.githubusercontent.com/datamisc/ts-2020/main/data.csv"
anes_data = pd.read_csv(data_url, compression='gzip')

vars = {
    "V201033": "vote_intention",
    "V201507x": "age",
    "V201600": "sex",
    "V201511x": "education",
    "V201617x": "income",
    "V201228": "party_id",
    "V201231x": "party_id_str",
    "V201232": "party_id_imp",
    "V201200": "ideology",
    "V201156": "feeling_democrat",
    "V201157": "feeling_republican",
    # "V201641": "political_knowledge_intro",
    # "V201642": "political_knowledge_catch1",
    # "V201643": "political_knowledge_catch_feedback",
    "V201644": "political_knowledge_senate_term",
    "V201645": "political_knowledge_least_spending",
    "V201646": "political_knowledge_house_majority",
    "V201647": "political_knowledge_senate_majority",
    "V202406": "political_interest",
    "V202407": "follow_politics_media",
    "V202408": "understand_issues",
    "V203001": 'state',
}

df = anes_data[vars.keys()]
df = df.rename(columns=vars)
df = df[df['age'] >= 18]
df = df[df['sex'].between(1,2)]
df['female'] = df['sex'].map({1: 0, 2: 1})
df = df[df['education'] > 0]
df = df[df['ideology'].between(1,7)]
df = df[df['party_id'].between(1,3)]
df = df[df['vote_intention'].between(1,2)]
df = df[df['political_interest'] > 0]
df = df[df['follow_politics_media'] > 0]
df = df[df['understand_issues'] > 0]
df = df[df['party_id_imp'] > 0 ]
df = df[(df['feeling_democrat'] >= 0) & (df['feeling_republican'] >= 0 )]
df['affective_polarization'] = (df['feeling_democrat'] - df['feeling_republican']).abs()

political_knowledge_vars = [
    "political_knowledge_senate_term",
    "political_knowledge_least_spending",
    "political_knowledge_house_majority",
    "political_knowledge_senate_majority"
]

df['political_knowledge_senate_term'] = clean_knowledge_variable(df['political_knowledge_senate_term'], [6])
df['political_knowledge_least_spending'] = clean_knowledge_variable(df['political_knowledge_least_spending'], [1])
df['political_knowledge_house_majority'] = clean_knowledge_variable(df['political_knowledge_house_majority'], [1])
df['political_knowledge_senate_majority'] = clean_knowledge_variable(df['political_knowledge_senate_majority'], [2])
df['political_knowledge_scale'] = df[political_knowledge_vars].sum(axis=1)
df = df.drop(columns=political_knowledge_vars)

In [ ]:
# Take a quicklook at the data


In [ ]:
# Some Models
# Use the cleaned data stored in `df` to some regression model.

# Define the model formula
# It takes the following form DV ~ IVs

form_ap = "affective_polarization ~ political_knowledge_scale"
form_ap_all = "affective_polarization ~ age + education + ideology + political_knowledge_scale + female"
form_identity_imp = "party_id_imp ~ political_knowledge_scale"
form_dem_ap = "feeling_democrat ~ political_knowledge_scale"
form_rep_ap = "feeling_republican ~ political_knowledge_scale"

In [ ]:
# Fit the regression model
models = [
  sm.ols(formula=form_ap, data=df).fit(),
  sm.ols(formula=form_ap_all, data=df).fit(),
  sm.ols(formula=form_identity_imp, data=df).fit(),
  sm.ols(formula=form_dem_ap, data=df).fit(),
  sm.ols(formula=form_rep_ap, data=df).fit(),
]

In [ ]:
# Print the summary of one of the regression models


In [ ]:
# Print the summary of different models using `summary_col`


In [ ]:
# Step 5: Visualize Results
# Create a visualization to summarize the results of the regression model.

# Function to create a DataFrame with coefficients and confidence intervals
def make_coefs(model):
    return pd.DataFrame({
        'coef': model.params,
        'lower_ci': model.conf_int()[0],
        'upper_ci': model.conf_int()[1],
        'pval': model.pvalues
    }).drop('Intercept')

coef_df = make_coefs(models[1])
coef_df


In [ ]:
# Make the figure
plt.figure(figsize=(8, 4))
# Plot each coefficient with its confidence interval
plt.errorbar(coef_df['coef'], coef_df.index, xerr=(coef_df['coef'] - coef_df['lower_ci'], coef_df['upper_ci'] - coef_df['coef']), fmt='o', color='b', elinewidth=2, capsize=4)
plt.axvline(x=0, color='grey', linestyle='--')  # Add a vertical line at zero for reference
plt.title('Regression Coefficients with Confidence Intervals')
plt.xlabel('Coefficient')
plt.ylabel('Variables')
plt.yticks(ticks=range(len(coef_df)), labels=coef_df.index)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()

### Hack-Time
Try to use the code above that creates the figure and turn it into a function that you can re-use with other models

In [ ]:
# Hack-Time


### What about Altair? 

Take some time to explore the Altair page: https://altair-viz.github.io/

### Hack-Time 
1. Find the installation instruction for altair and install the library.
2. Copy-paste some code from the website and try to make a figure 
3. Make a figure using altair and data from our `df`

In [ ]:
# Hack-Time